In [17]:
from pathlib import Path
import re
import h5py
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

print("Imports ready")

Imports ready


In [ ]:
# case_0009/postproc 기준으로 TXT + H5를 자동 선택합니다.
from postproc_interop.header_mapping import (
    MOTORCAD_TXT_HEADER_TO_H5_KEY_MAP,
    map_header_tokens_to_h5_keys,
)

workspace_root = Path("d:/KDH/NvidiaNemo")
case_postproc_dir = Path("D:/KDH/Sim_4SolverX/DOE_TrainingData/case_0009/postproc")

if not case_postproc_dir.exists():
    raise FileNotFoundError(f"Folder not found: {case_postproc_dir}")

txt_candidates = sorted(case_postproc_dir.glob("*.txt"))
h5_candidates = sorted(case_postproc_dir.glob("*.h5"))

if not h5_candidates:
    raise FileNotFoundError(f"No .h5 found under: {case_postproc_dir}")

# 우선순위: StaticLoad -> OnLoadTorque -> 첫 번째
priority = ["StaticLoad", "OnLoadTorque"]


def _pick_h5(paths):
    for p in paths:
        for k in priority:
            if k.lower() in p.name.lower():
                return p
    return paths[0]


h5_path = _pick_h5(h5_candidates)
txt_path = txt_candidates[1] if txt_candidates else None

raw_lines = []
if txt_path and txt_path.exists():
    raw_lines = txt_path.read_text(
        encoding="utf-8", errors="ignore"
    ).splitlines()

lines = []
for line in raw_lines:
    s = line.strip()
    if not s:
        continue
    if s.startswith("#") or s.startswith("//") or s.startswith(";"):
        continue
    lines.append(s)

# TXT에 h5 경로 문자열이 있으면 우선 사용
h5_pattern = re.compile(r"[A-Za-z]:\\[^\n\r\t\"]+\.h5|/[^\n\r\t\"]+\.h5")
for s in lines:
    found = h5_pattern.findall(s)
    if found:
        candidate = Path(found[0])
        if candidate.exists():
            h5_path = candidate
            break

# TXT 라인에서 h5 키 후보 추출 (단위 문자열 제외)
# 허용 예: mesh/node_1, fields/bx, regions/name
h5_key_pattern = re.compile(r"^[A-Za-z0-9_]+/[A-Za-z0-9_]+(?:/[A-Za-z0-9_]+)*$")
key_like = []
for s in lines:
    candidate = s.strip().strip(",")
    if h5_key_pattern.match(candidate):
        key_like.append(candidate)

# TXT 헤더 토큰 추출 (예: TriIndex, Node1, Node2, ...)
header_tokens = []
for s in lines:
    if "," in s and not "/" in s:
        parts = [p.strip() for p in s.split(",") if p.strip()]
        has_alpha = any(
            any(ch.isalpha() for ch in p) for p in parts
        )
        if has_alpha:
            header_tokens.extend(parts)

txt_header_tokens = sorted(set(header_tokens))

# 공통 매핑 사전을 사용해 TXT 헤더를 H5 키로 변환
header_to_h5_map = MOTORCAD_TXT_HEADER_TO_H5_KEY_MAP
mapped_keys_from_header = map_header_tokens_to_h5_keys(txt_header_tokens)

print("Case folder:", case_postproc_dir)
print("TXT selected:", txt_path)
print("H5 selected:", h5_path)
print("TXT parsed lines:", len(lines))
print("TXT slash-style keys:", len(key_like))
print("TXT header tokens:", len(txt_header_tokens))
print(
    "Mapped keys from TXT headers:",
    sorted(set(mapped_keys_from_header)),
)

Case folder: D:\KDH\Sim_4SolverX\DOE_TrainingData\case_0009\postproc
TXT selected: D:\KDH\Sim_4SolverX\DOE_TrainingData\case_0009\postproc\Mag_OnLoadTorque_result_1.txt
H5 selected: D:\KDH\Sim_4SolverX\DOE_TrainingData\case_0009\postproc\Mag_OnLoadTorque_result_1.h5
TXT parsed lines: 920556
TXT slash-style keys: 0
TXT header tokens: 530
Mapped keys from TXT headers: ['fields/a', 'fields/bx', 'fields/by', 'fields/j', 'mesh/node_1', 'mesh/node_2', 'mesh/node_3', 'mesh/node_id', 'mesh/node_x_mm', 'mesh/node_y_mm', 'mesh/reg_code', 'mesh/tri_index', 'regions/name', 'regions/reg_code']


In [16]:
# 9. H5 -> nodetable / regiontable / elementtable in-memory mapping (shared helper)

from postproc_interop.tabular import load_h5_as_tables

(
    meshframe,
    nodetable,
    regiontable,
    elementtable,
    elementtable_with_xy,
    table_stats,
) = load_h5_as_tables(h5_path)

missing_n1 = table_stats["missing_n1"]
missing_n2 = table_stats["missing_n2"]
missing_n3 = table_stats["missing_n3"]

print("meshframe summary:", meshframe.summary())
print("nodetable shape:", nodetable.shape)
print("elementtable shape:", elementtable.shape)
print("regiontable shape:", None if regiontable is None else regiontable.shape)
print("missing element-node refs (Node1, Node2, Node3):", (missing_n1, missing_n2, missing_n3))

print("\n[nodetable] head")
display(nodetable.head(10))

if regiontable is None:
    print("\n[regiontable] None (regions/reg_code or regions/name not found)")
else:
    print("\n[regiontable] head")
    display(regiontable.head(10))

print("\n[elementtable] head")
display(elementtable.head(10))

print("\n[elementtable_with_xy] head")
display(elementtable_with_xy.head(10))

meshframe summary: {'n_elements': 13344, 'n_nodes': 6944, 'field_keys': ['a', 'bx', 'by', 'j'], 'has_regions': True}
nodetable shape: (6944, 3)
elementtable shape: (13344, 5)
regiontable shape: (69, 2)
missing element-node refs (Node1, Node2, Node3): (0, 0, 0)

[nodetable] head


,NodeIndex,X,Y
0,1,9.484000,-20.812000
1,2,0.000000,0.000000
2,3,78.862000,-3.071000
3,4,16.726999,-15.598000
4,5,80.080002,-3.186000
5,6,21.423000,-8.010000
6,7,28.415001,-62.351002
7,8,81.279999,-3.230000
8,9,50.112000,-46.730999
9,10,80.362000,-3.175000



[regiontable] head


,RegionCode,RegionName
0,1,Stator
1,2,ArmatureSlotF6
2,3,ArmatureSlotA6
3,4,ArmatureSlotB6
4,5,ArmatureSlotC6
5,6,ArmatureSlotD6
6,7,ArmatureSlotE6
7,8,StatorWedge
8,9,StatorAir
9,10,Rotor



[elementtable] head


,TriIndex,Node1,Node2,Node3,RegCode
0,4871,210,619,1338,1
1,4872,618,617,3914,1
2,4873,615,614,3915,1
3,4874,612,611,3916,1
4,4875,609,608,3917,1
5,4876,213,1336,3918,1
6,4877,1334,3919,4027,1
7,4878,1333,3920,4011,1
8,4879,1332,217,3921,1
9,4880,1330,3922,4013,1



[elementtable_with_xy] head


,TriIndex,Node1,Node2,Node3,RegCode,Node1_X,Node1_Y,Node2_X,Node2_Y,Node3_X,Node3_Y
0,4871,210,619,1338,1,55.153999,-42.320999,54.973000,-42.556000,55.804001,-42.820000
1,4872,618,617,3914,1,54.791000,-42.790001,54.608002,-43.022999,54.884998,-43.334999
2,4873,615,614,3915,1,54.240002,-43.487000,54.054001,-43.717999,54.268002,-44.012001
3,4874,612,611,3916,1,53.679001,-44.176998,53.490002,-44.404999,53.734001,-44.720001
4,4875,609,608,3917,1,53.110001,-44.860001,52.917999,-45.085999,53.070999,-45.230000
5,4876,213,1336,3918,1,53.477001,-45.970001,53.889000,-45.639999,53.498001,-45.250000
6,4877,1334,3919,4027,1,55.311001,-45.501999,56.123001,-45.126999,55.269001,-44.558998
7,4878,1333,3920,4011,1,57.366001,-47.304001,58.076000,-46.762001,57.178001,-45.849998
8,4879,1332,217,3921,1,59.133999,-48.854000,60.018002,-49.630001,60.439999,-48.926998
9,4880,1330,3922,4013,1,62.669998,-51.956001,63.766998,-51.287998,62.397999,-50.602001
